18.2 — Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_DIR = Path(
    r"C:\Users\acer\Desktop\ProgettoTesi"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "risultati"
)

EMBEDDING_DIR = (
    RESULTS_DIR
    / "embedding_multimodali_turn_level"
)

CLUSTER_DIR = (
    RESULTS_DIR
    / "clustering_embedding_multimodali"
)

CLUSTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)


METADATA_PATH = (
    EMBEDDING_DIR
    / "metadata_turni_multimodali.csv"
)

AUDIO_PATH = (
    EMBEDDING_DIR
    / "embedding_audio_whisper_small_l2.npy"
)

TEXT_PATH = (
    EMBEDDING_DIR
    / "embedding_testuali_multilingual_minilm_l2.npy"
)

FUSION_PATH = (
    EMBEDDING_DIR
    / "embedding_fusion_audio_testo_early_equal_weight.npy"
)


metadata_df = pd.read_csv(
    METADATA_PATH
)

X_audio = np.load(
    AUDIO_PATH
)

X_text = np.load(
    TEXT_PATH
)

X_fusion = np.load(
    FUSION_PATH
)


print("Metadata:", metadata_df.shape)
print("Audio:", X_audio.shape)
print("Testo:", X_text.shape)
print("Fusion:", X_fusion.shape)

print(
    "\nPazienti:",
    metadata_df["patient_id"].nunique()
)

Metadata: (3710, 7)
Audio: (3710, 768)
Testo: (3710, 384)
Fusion: (3710, 1152)

Pazienti: 90


18.3 — QC rappresentazioni

In [2]:
assert len(metadata_df) == X_audio.shape[0]
assert len(metadata_df) == X_text.shape[0]
assert len(metadata_df) == X_fusion.shape[0]

assert metadata_df["turn_id"].nunique() == len(metadata_df)

for name, X in [
    ("audio", X_audio),
    ("text", X_text),
    ("fusion", X_fusion)
]:

    print(
        name,
        "| NaN:",
        int(np.isnan(X).sum()),
        "| Inf:",
        int(np.isinf(X).sum())
    )

print("\n✓ Rappresentazioni correttamente allineate.")

audio | NaN: 0 | Inf: 0
text | NaN: 0 | Inf: 0
fusion | NaN: 0 | Inf: 0

✓ Rappresentazioni correttamente allineate.


18.4 — Campione bilanciato comune alle tre rappresentazioni

In [3]:
MAX_TURNS_PER_PATIENT = 25
RANDOM_STATE = 42


balanced_indices = (
    metadata_df
    .groupby(
        "patient_id",
        group_keys=False
    )
    .apply(
        lambda group:
            group.sample(
                n=min(
                    len(group),
                    MAX_TURNS_PER_PATIENT
                ),
                random_state=RANDOM_STATE
            )
    )
    .index
    .to_numpy()
)


balanced_indices = np.sort(
    balanced_indices
)


balanced_metadata = (
    metadata_df
    .loc[balanced_indices]
    .copy()
    .reset_index(drop=True)
)


X_audio_bal = X_audio[
    balanced_indices
]

X_text_bal = X_text[
    balanced_indices
]

X_fusion_bal = X_fusion[
    balanced_indices
]


print(
    "Turni originali:",
    len(metadata_df)
)

print(
    "Turni nel fit bilanciato:",
    len(balanced_metadata)
)

print(
    "Pazienti:",
    balanced_metadata[
        "patient_id"
    ].nunique()
)


patient_counts = (
    balanced_metadata
    .groupby("patient_id")
    .size()
)


display(
    patient_counts.describe()
)

Turni originali: 3710
Turni nel fit bilanciato: 2156
Pazienti: 90


C:\Users\acer\AppData\Local\Temp\ipykernel_25328\125428123.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


count    90.000000
mean     23.955556
std       3.259941
min       8.000000
25%      25.000000
50%      25.000000
75%      25.000000
max      25.000000
dtype: float64

18.5 — PCA delle rappresentazioni

In [4]:
from sklearn.decomposition import PCA


def fit_pca_95(X, name):

    pca = PCA(
        n_components=0.95,
        svd_solver="full",
        random_state=42
    )

    X_pca = pca.fit_transform(
        X
    )

    print(
        f"{name:10s} | "
        f"{X.shape[1]} -> {X_pca.shape[1]} componenti | "
        f"varianza={pca.explained_variance_ratio_.sum():.4f}"
    )

    return pca, X_pca


pca_audio, X_audio_pca = fit_pca_95(
    X_audio_bal,
    "AUDIO"
)

pca_text, X_text_pca = fit_pca_95(
    X_text_bal,
    "TESTO"
)

pca_fusion, X_fusion_pca = fit_pca_95(
    X_fusion_bal,
    "FUSION"
)

AUDIO      | 768 -> 266 componenti | varianza=0.9500
TESTO      | 384 -> 137 componenti | varianza=0.9502
FUSION     | 1152 -> 263 componenti | varianza=0.9502


18.6 — Rappresentazioni pronte per il clustering

In [ ]:
REPRESENTATIONS = {
    "audio": X_audio_pca,
    "text": X_text_pca,
    "fusion": X_fusion_pca
}


for name, X in REPRESENTATIONS.items():

    print(
        name,
        "->",
        X.shape
    )

audio -> (2156, 266)
text -> (2156, 137)
fusion -> (2156, 263)


18.7 — Metriche comuni di clustering

In [6]:
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)


def evaluate_clustering(
    X,
    labels,
    representation,
    algorithm,
    k=None
):

    unique_labels = np.unique(labels)

    # HDBSCAN verrà gestito successivamente:
    # qui assumiamo che tutti i punti appartengano a un cluster.
    n_clusters = len(unique_labels)

    counts = pd.Series(labels).value_counts()

    if n_clusters < 2:
        return {
            "representation": representation,
            "algorithm": algorithm,
            "k": k,
            "n_clusters": n_clusters,
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "smallest_cluster_n": int(counts.min()),
            "largest_cluster_n": int(counts.max())
        }

    return {
        "representation": representation,
        "algorithm": algorithm,
        "k": k,
        "n_clusters": n_clusters,

        "silhouette":
            silhouette_score(
                X,
                labels
            ),

        "calinski_harabasz":
            calinski_harabasz_score(
                X,
                labels
            ),

        "davies_bouldin":
            davies_bouldin_score(
                X,
                labels
            ),

        "smallest_cluster_n":
            int(counts.min()),

        "largest_cluster_n":
            int(counts.max())
    }

18.8 — K-means

In [7]:
from sklearn.cluster import KMeans
import time


K_VALUES = range(2, 7)

kmeans_results = []
kmeans_labels = {}
kmeans_models = {}


for representation, X in REPRESENTATIONS.items():

    print(
        f"\n===== K-MEANS | {representation.upper()} ====="
    )

    for k in K_VALUES:

        start = time.time()

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=50
        )

        labels = model.fit_predict(X)

        metrics = evaluate_clustering(
            X=X,
            labels=labels,
            representation=representation,
            algorithm="KMeans",
            k=k
        )

        metrics["inertia"] = model.inertia_
        metrics["elapsed_seconds"] = (
            time.time() - start
        )

        kmeans_results.append(
            metrics
        )

        kmeans_labels[
            (representation, k)
        ] = labels

        kmeans_models[
            (representation, k)
        ] = model

        print(
            f"K={k} | "
            f"sil={metrics['silhouette']:.4f} | "
            f"CH={metrics['calinski_harabasz']:.2f} | "
            f"DB={metrics['davies_bouldin']:.4f} | "
            f"min={metrics['smallest_cluster_n']} | "
            f"max={metrics['largest_cluster_n']} | "
            f"{metrics['elapsed_seconds']:.1f}s"
        )


kmeans_results_df = pd.DataFrame(
    kmeans_results
)

display(
    kmeans_results_df.round(4)
)


===== K-MEANS | AUDIO =====
K=2 | sil=0.2404 | CH=791.87 | DB=1.5408 | min=767 | max=1389 | 3.6s
K=3 | sil=0.1660 | CH=592.86 | DB=1.8315 | min=377 | max=1046 | 0.8s
K=4 | sil=0.1210 | CH=464.74 | DB=2.1694 | min=251 | max=802 | 1.0s
K=5 | sil=0.0958 | CH=381.38 | DB=2.4637 | min=166 | max=602 | 1.3s
K=6 | sil=0.0996 | CH=330.84 | DB=2.4217 | min=148 | max=538 | 1.5s

===== K-MEANS | TEXT =====
K=2 | sil=0.0785 | CH=157.99 | DB=3.6632 | min=994 | max=1162 | 0.3s
K=3 | sil=0.0752 | CH=121.53 | DB=3.7080 | min=286 | max=1004 | 0.5s
K=4 | sil=0.0477 | CH=108.55 | DB=3.4279 | min=245 | max=697 | 0.6s
K=5 | sil=0.0534 | CH=100.14 | DB=3.3024 | min=238 | max=654 | 0.7s
K=6 | sil=0.0456 | CH=92.43 | DB=3.2652 | min=206 | max=523 | 0.9s

===== K-MEANS | FUSION =====
K=2 | sil=0.1103 | CH=252.52 | DB=2.8715 | min=869 | max=1287 | 0.5s
K=3 | sil=0.0684 | CH=172.68 | DB=3.5951 | min=428 | max=936 | 0.8s
K=4 | sil=0.0451 | CH=136.07 | DB=3.6215 | min=407 | max=729 | 1.1s
K=5 | sil=0.0490 | CH=117

,representation,algorithm,k,n_clusters,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n,inertia,elapsed_seconds
0,audio,KMeans,2,2,0.2404,791.8741,1.5408,767,1389,386.2786,3.6292
1,audio,KMeans,3,3,0.1660,592.8625,1.8315,377,1046,340.6689,0.7900
2,audio,KMeans,4,4,0.1210,464.7409,2.1694,251,802,320.5867,0.9533
3,audio,KMeans,5,5,0.0958,381.3809,2.4637,166,602,309.0809,1.3200
4,audio,KMeans,6,6,0.0996,330.8446,2.4217,148,538,298.5668,1.5143
5,text,KMeans,2,2,0.0785,157.9928,3.6632,994,1162,1330.1816,0.3316
6,text,KMeans,3,3,0.0752,121.5263,3.7080,286,1004,1282.9192,0.4706
7,text,KMeans,4,4,0.0477,108.5502,3.4279,245,697,1240.0923,0.5795
8,text,KMeans,5,5,0.0534,100.1399,3.3024,238,654,1203.6119,0.7323
9,text,KMeans,6,6,0.0456,92.4265,3.2652,206,523,1175.1548,0.8679


18.9 — Gaussian Mixture Model

In [8]:
from sklearn.mixture import GaussianMixture


gmm_results = []
gmm_labels = {}
gmm_models = {}
gmm_probabilities = {}


for representation, X in REPRESENTATIONS.items():

    print(
        f"\n===== GMM | {representation.upper()} ====="
    )

    for k in K_VALUES:

        start = time.time()

        model = GaussianMixture(
            n_components=k,
            covariance_type="diag",
            random_state=42,
            n_init=3,
            max_iter=500,
            reg_covar=1e-6
        )

        labels = model.fit_predict(X)

        probabilities = (
            model.predict_proba(X)
        )

        metrics = evaluate_clustering(
            X=X,
            labels=labels,
            representation=representation,
            algorithm="GMM",
            k=k
        )

        metrics["aic"] = model.aic(X)
        metrics["bic"] = model.bic(X)
        metrics["converged"] = model.converged_
        metrics["elapsed_seconds"] = (
            time.time() - start
        )

        # Quanto è netta, in media,
        # l'appartenenza al cluster più probabile.
        metrics[
            "mean_max_membership_probability"
        ] = (
            probabilities
            .max(axis=1)
            .mean()
        )

        gmm_results.append(
            metrics
        )

        gmm_labels[
            (representation, k)
        ] = labels

        gmm_models[
            (representation, k)
        ] = model

        gmm_probabilities[
            (representation, k)
        ] = probabilities

        print(
            f"K={k} | "
            f"sil={metrics['silhouette']:.4f} | "
            f"CH={metrics['calinski_harabasz']:.2f} | "
            f"DB={metrics['davies_bouldin']:.4f} | "
            f"BIC={metrics['bic']:.1f} | "
            f"Pmax={metrics['mean_max_membership_probability']:.3f} | "
            f"min={metrics['smallest_cluster_n']} | "
            f"max={metrics['largest_cluster_n']} | "
            f"conv={metrics['converged']} | "
            f"{metrics['elapsed_seconds']:.1f}s"
        )


gmm_results_df = pd.DataFrame(
    gmm_results
)

display(
    gmm_results_df.round(4)
)


===== GMM | AUDIO =====
K=2 | sil=0.1469 | CH=447.97 | DB=2.1336 | BIC=-3158942.8 | Pmax=0.987 | min=941 | max=1215 | conv=True | 0.2s
K=3 | sil=0.0912 | CH=388.93 | DB=2.5326 | BIC=-3163678.7 | Pmax=0.984 | min=338 | max=1053 | conv=True | 0.3s
K=4 | sil=0.0619 | CH=303.19 | DB=3.3937 | BIC=-3163432.7 | Pmax=0.973 | min=245 | max=902 | conv=True | 0.6s
K=5 | sil=0.0477 | CH=269.24 | DB=3.9998 | BIC=-3162271.6 | Pmax=0.973 | min=162 | max=731 | conv=True | 0.6s
K=6 | sil=0.0637 | CH=280.28 | DB=3.6911 | BIC=-3160329.0 | Pmax=0.978 | min=93 | max=651 | conv=True | 0.7s

===== GMM | TEXT =====
K=2 | sil=0.0168 | CH=47.71 | DB=6.4750 | BIC=-932577.4 | Pmax=0.988 | min=906 | max=1250 | conv=True | 0.3s
K=3 | sil=-0.0254 | CH=30.33 | DB=8.4661 | BIC=-938608.8 | Pmax=0.978 | min=365 | max=995 | conv=True | 0.8s
K=4 | sil=-0.0453 | CH=25.96 | DB=7.4367 | BIC=-943409.6 | Pmax=0.976 | min=54 | max=1035 | conv=True | 0.7s
K=5 | sil=-0.0386 | CH=29.35 | DB=6.2892 | BIC=-943754.7 | Pmax=0.978 | m

,representation,algorithm,k,n_clusters,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n,aic,bic,converged,elapsed_seconds,mean_max_membership_probability
0,audio,GMM,2,2,0.1469,447.9700,2.1336,941,1215,-3.164988e+06,-3.158943e+06,True,0.2302,0.9867
1,audio,GMM,3,3,0.0912,388.9280,2.5326,338,1053,-3.172749e+06,-3.163679e+06,True,0.3401,0.9838
2,audio,GMM,4,4,0.0619,303.1919,3.3937,245,902,-3.175528e+06,-3.163433e+06,True,0.6062,0.9733
3,audio,GMM,5,5,0.0477,269.2425,3.9998,162,731,-3.177392e+06,-3.162272e+06,True,0.6220,0.9725
4,audio,GMM,6,6,0.0637,280.2847,3.6911,93,651,-3.178475e+06,-3.160329e+06,True,0.7060,0.9781
5,text,GMM,2,2,0.0168,47.7142,6.4750,906,1250,-9.356935e+05,-9.325774e+05,True,0.2826,0.9880
6,text,GMM,3,3,-0.0254,30.3317,8.4661,365,995,-9.432859e+05,-9.386088e+05,True,0.7725,0.9776
7,text,GMM,4,4,-0.0453,25.9649,7.4367,54,1035,-9.496476e+05,-9.434096e+05,True,0.6559,0.9759
8,text,GMM,5,5,-0.0386,29.3494,6.2892,54,965,-9.515536e+05,-9.437547e+05,True,0.5742,0.9779
9,text,GMM,6,6,-0.0607,33.2262,5.7727,49,870,-9.511574e+05,-9.417976e+05,True,1.1734,0.9686


18.10 — Confronto iniziale

In [9]:
initial_clustering_results = pd.concat(
    [
        kmeans_results_df,
        gmm_results_df
    ],
    ignore_index=True,
    sort=False
)


comparison_view = (
    initial_clustering_results[
        [
            "representation",
            "algorithm",
            "k",
            "silhouette",
            "calinski_harabasz",
            "davies_bouldin",
            "smallest_cluster_n",
            "largest_cluster_n"
        ]
    ]
    .sort_values(
        "silhouette",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    comparison_view.round(4)
)

,representation,algorithm,k,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n
0,audio,KMeans,2,0.2404,791.8741,1.5408,767,1389
1,audio,KMeans,3,0.1660,592.8625,1.8315,377,1046
2,audio,GMM,2,0.1469,447.9700,2.1336,941,1215
3,audio,KMeans,4,0.1210,464.7409,2.1694,251,802
4,fusion,KMeans,2,0.1103,252.5158,2.8715,869,1287
5,audio,KMeans,6,0.0996,330.8446,2.4217,148,538
6,audio,KMeans,5,0.0958,381.3809,2.4637,166,602
7,audio,GMM,3,0.0912,388.9280,2.5326,338,1053
8,text,KMeans,2,0.0785,157.9928,3.6632,994,1162
9,text,KMeans,3,0.0752,121.5263,3.7080,286,1004


18.11 — Agglomerative clustering (Ward)

In [10]:
from sklearn.cluster import AgglomerativeClustering
import time


hierarchical_results = []
hierarchical_labels = {}


for representation, X in REPRESENTATIONS.items():

    print(
        f"\n===== GERARCHICO | {representation.upper()} ====="
    )

    for k in K_VALUES:

        start = time.time()

        model = AgglomerativeClustering(
            n_clusters=k,
            linkage="ward"
        )

        labels = model.fit_predict(X)

        metrics = evaluate_clustering(
            X=X,
            labels=labels,
            representation=representation,
            algorithm="Agglomerative_Ward",
            k=k
        )

        metrics["elapsed_seconds"] = (
            time.time() - start
        )

        hierarchical_results.append(
            metrics
        )

        hierarchical_labels[
            (representation, k)
        ] = labels

        print(
            f"K={k} | "
            f"sil={metrics['silhouette']:.4f} | "
            f"CH={metrics['calinski_harabasz']:.2f} | "
            f"DB={metrics['davies_bouldin']:.4f} | "
            f"min={metrics['smallest_cluster_n']} | "
            f"max={metrics['largest_cluster_n']} | "
            f"{metrics['elapsed_seconds']:.1f}s"
        )


hierarchical_results_df = pd.DataFrame(
    hierarchical_results
)

display(
    hierarchical_results_df.round(4)
)


===== GERARCHICO | AUDIO =====
K=2 | sil=0.2505 | CH=662.37 | DB=1.3352 | min=430 | max=1726 | 1.1s
K=3 | sil=0.1514 | CH=561.36 | DB=1.9931 | min=430 | max=953 | 0.6s
K=4 | sil=0.1305 | CH=425.20 | DB=1.8540 | min=106 | max=953 | 0.6s
K=5 | sil=0.0876 | CH=344.60 | DB=2.3837 | min=106 | max=953 | 0.6s
K=6 | sil=0.0901 | CH=295.89 | DB=2.2763 | min=47 | max=906 | 0.5s

===== GERARCHICO | TEXT =====
K=2 | sil=0.0174 | CH=84.57 | DB=4.4181 | min=656 | max=1500 | 0.4s
K=3 | sil=0.0195 | CH=77.20 | DB=3.8660 | min=114 | max=1386 | 0.4s
K=4 | sil=0.0276 | CH=72.34 | DB=3.7694 | min=114 | max=1206 | 0.4s
K=5 | sil=0.0063 | CH=69.31 | DB=3.4055 | min=114 | max=1206 | 0.4s
K=6 | sil=0.0157 | CH=67.32 | DB=3.6306 | min=114 | max=915 | 0.5s

===== GERARCHICO | FUSION =====
K=2 | sil=0.0799 | CH=189.59 | DB=3.3437 | min=1055 | max=1101 | 0.6s
K=3 | sil=0.0712 | CH=130.59 | DB=4.0004 | min=424 | max=1101 | 0.7s
K=4 | sil=0.0678 | CH=107.85 | DB=3.3266 | min=182 | max=1101 | 0.6s
K=5 | sil=0.0359 

,representation,algorithm,k,n_clusters,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n,elapsed_seconds
0,audio,Agglomerative_Ward,2,2,0.2505,662.3671,1.3352,430,1726,1.0711
1,audio,Agglomerative_Ward,3,3,0.1514,561.3632,1.9931,430,953,0.6176
2,audio,Agglomerative_Ward,4,4,0.1305,425.2021,1.8540,106,953,0.5581
3,audio,Agglomerative_Ward,5,5,0.0876,344.6005,2.3837,106,953,0.5532
4,audio,Agglomerative_Ward,6,6,0.0901,295.8917,2.2763,47,906,0.5170
5,text,Agglomerative_Ward,2,2,0.0174,84.5716,4.4181,656,1500,0.4216
6,text,Agglomerative_Ward,3,3,0.0195,77.2000,3.8660,114,1386,0.4235
7,text,Agglomerative_Ward,4,4,0.0276,72.3407,3.7694,114,1206,0.4097
8,text,Agglomerative_Ward,5,5,0.0063,69.3129,3.4055,114,1206,0.3784
9,text,Agglomerative_Ward,6,6,0.0157,67.3156,3.6306,114,915,0.4676


18.12 — Verifica HDBSCAN

In [11]:
import importlib.util

print(
    "Pacchetto hdbscan disponibile:",
    importlib.util.find_spec("hdbscan") is not None
)

Pacchetto hdbscan disponibile: False


In [12]:
import sys
import subprocess

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "hdbscan"
    ]
)

print("✓ hdbscan installato.")

✓ hdbscan installato.


18.13 — HDBSCAN

In [13]:
import hdbscan
import time

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)


MIN_CLUSTER_SIZES = [
    30,
    50,
    100
]


hdbscan_results = []
hdbscan_labels = {}
hdbscan_models = {}


for representation, X in REPRESENTATIONS.items():

    print(
        f"\n===== HDBSCAN | {representation.upper()} ====="
    )

    for min_cluster_size in MIN_CLUSTER_SIZES:

        start = time.time()

        model = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=None,
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=True
        )

        labels = model.fit_predict(X)

        non_noise = labels != -1

        n_clusters = len(
            set(labels) - {-1}
        )

        n_noise = int(
            (labels == -1).sum()
        )

        noise_fraction = (
            n_noise / len(labels)
        )


        if (
            n_clusters >= 2
            and
            non_noise.sum() > n_clusters
        ):

            X_valid = X[
                non_noise
            ]

            labels_valid = labels[
                non_noise
            ]

            silhouette = silhouette_score(
                X_valid,
                labels_valid
            )

            ch = calinski_harabasz_score(
                X_valid,
                labels_valid
            )

            db = davies_bouldin_score(
                X_valid,
                labels_valid
            )

            counts = (
                pd.Series(
                    labels_valid
                )
                .value_counts()
            )

            smallest = int(
                counts.min()
            )

            largest = int(
                counts.max()
            )

        else:

            silhouette = np.nan
            ch = np.nan
            db = np.nan
            smallest = np.nan
            largest = np.nan


        row = {
            "representation":
                representation,

            "algorithm":
                "HDBSCAN",

            "min_cluster_size":
                min_cluster_size,

            "n_clusters":
                n_clusters,

            "n_noise":
                n_noise,

            "noise_fraction":
                noise_fraction,

            "silhouette":
                silhouette,

            "calinski_harabasz":
                ch,

            "davies_bouldin":
                db,

            "smallest_cluster_n":
                smallest,

            "largest_cluster_n":
                largest,

            "mean_membership_probability":
                (
                    model.probabilities_[
                        non_noise
                    ].mean()
                    if non_noise.any()
                    else np.nan
                ),

            "elapsed_seconds":
                time.time() - start
        }


        hdbscan_results.append(
            row
        )

        hdbscan_labels[
            (
                representation,
                min_cluster_size
            )
        ] = labels

        hdbscan_models[
            (
                representation,
                min_cluster_size
            )
        ] = model


        print(
            f"min_cluster={min_cluster_size} | "
            f"clusters={n_clusters} | "
            f"noise={noise_fraction:.1%} | "
            f"sil={silhouette:.4f} | "
            f"CH={ch:.2f} | "
            f"DB={db:.4f} | "
            f"{row['elapsed_seconds']:.1f}s"
        )


hdbscan_results_df = pd.DataFrame(
    hdbscan_results
)


display(
    hdbscan_results_df.round(4)
)


===== HDBSCAN | AUDIO =====
min_cluster=30 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 5.7s
min_cluster=50 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 5.9s
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 5.8s

===== HDBSCAN | TEXT =====
min_cluster=30 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 3.0s
min_cluster=50 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 3.4s
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 3.4s

===== HDBSCAN | FUSION =====
min_cluster=30 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 6.7s
min_cluster=50 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 6.4s
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan | 6.7s


,representation,algorithm,min_cluster_size,n_clusters,n_noise,noise_fraction,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n,mean_membership_probability,elapsed_seconds
0,audio,HDBSCAN,30,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,5.7398
1,audio,HDBSCAN,50,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,5.9249
2,audio,HDBSCAN,100,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,5.7546
3,text,HDBSCAN,30,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0148
4,text,HDBSCAN,50,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.4269
5,text,HDBSCAN,100,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.4135
6,fusion,HDBSCAN,30,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,6.6850
7,fusion,HDBSCAN,50,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,6.3529
8,fusion,HDBSCAN,100,0,2156,1.0,NaN,NaN,NaN,NaN,NaN,NaN,6.6642


18.14 — Sensitivity HDBSCAN su PCA-50

In [14]:
N_COMPONENTS_HDBSCAN = 50


HDBSCAN_REPRESENTATIONS = {
    "audio": X_audio_pca[:, :N_COMPONENTS_HDBSCAN],
    "text": X_text_pca[:, :N_COMPONENTS_HDBSCAN],
    "fusion": X_fusion_pca[:, :N_COMPONENTS_HDBSCAN]
}


print("Varianza conservata dalle prime 50 componenti:")

print(
    "AUDIO:",
    round(
        pca_audio.explained_variance_ratio_[
            :N_COMPONENTS_HDBSCAN
        ].sum(),
        4
    )
)

print(
    "TESTO:",
    round(
        pca_text.explained_variance_ratio_[
            :N_COMPONENTS_HDBSCAN
        ].sum(),
        4
    )
)

print(
    "FUSION:",
    round(
        pca_fusion.explained_variance_ratio_[
            :N_COMPONENTS_HDBSCAN
        ].sum(),
        4
    )
)

Varianza conservata dalle prime 50 componenti:
AUDIO: 0.7946
TESTO: 0.7484
FUSION: 0.6887


HDBSCAN PCA-50

In [15]:
HDBSCAN_MIN_CLUSTER_SIZES = [
    30,
    50,
    100
]

HDBSCAN_MIN_SAMPLES = 5


hdbscan_pca50_results = []
hdbscan_pca50_labels = {}


for representation, X in HDBSCAN_REPRESENTATIONS.items():

    print(
        f"\n===== HDBSCAN PCA-50 | {representation.upper()} ====="
    )

    for min_cluster_size in HDBSCAN_MIN_CLUSTER_SIZES:

        start = time.time()

        model = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_method="eom"
        )

        labels = model.fit_predict(X)

        non_noise = labels != -1

        n_clusters = len(
            set(labels) - {-1}
        )

        n_noise = int(
            (labels == -1).sum()
        )

        noise_fraction = (
            n_noise / len(labels)
        )


        if (
            n_clusters >= 2
            and
            non_noise.sum() > n_clusters
        ):

            X_valid = X[non_noise]
            labels_valid = labels[non_noise]

            silhouette = silhouette_score(
                X_valid,
                labels_valid
            )

            ch = calinski_harabasz_score(
                X_valid,
                labels_valid
            )

            db = davies_bouldin_score(
                X_valid,
                labels_valid
            )

            counts = (
                pd.Series(labels_valid)
                .value_counts()
            )

            smallest = int(
                counts.min()
            )

            largest = int(
                counts.max()
            )

        else:

            silhouette = np.nan
            ch = np.nan
            db = np.nan
            smallest = np.nan
            largest = np.nan


        row = {
            "representation":
                representation,

            "algorithm":
                "HDBSCAN_PCA50",

            "min_cluster_size":
                min_cluster_size,

            "min_samples":
                HDBSCAN_MIN_SAMPLES,

            "n_clusters":
                n_clusters,

            "n_noise":
                n_noise,

            "noise_fraction":
                noise_fraction,

            "silhouette":
                silhouette,

            "calinski_harabasz":
                ch,

            "davies_bouldin":
                db,

            "smallest_cluster_n":
                smallest,

            "largest_cluster_n":
                largest,

            "elapsed_seconds":
                time.time() - start
        }


        hdbscan_pca50_results.append(
            row
        )

        hdbscan_pca50_labels[
            (
                representation,
                min_cluster_size
            )
        ] = labels


        print(
            f"min_cluster={min_cluster_size} | "
            f"clusters={n_clusters} | "
            f"noise={noise_fraction:.1%} | "
            f"sil={silhouette:.4f} | "
            f"CH={ch:.2f} | "
            f"DB={db:.4f}"
        )


hdbscan_pca50_results_df = pd.DataFrame(
    hdbscan_pca50_results
)

display(
    hdbscan_pca50_results_df.round(4)
)


===== HDBSCAN PCA-50 | AUDIO =====
min_cluster=30 | clusters=2 | noise=61.1% | sil=0.1060 | CH=30.60 | DB=1.5382
min_cluster=50 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan

===== HDBSCAN PCA-50 | TEXT =====
min_cluster=30 | clusters=5 | noise=74.0% | sil=0.2716 | CH=97.85 | DB=1.2724
min_cluster=50 | clusters=2 | noise=58.7% | sil=0.2514 | CH=111.41 | DB=1.2839
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan

===== HDBSCAN PCA-50 | FUSION =====
min_cluster=30 | clusters=3 | noise=77.6% | sil=0.2584 | CH=104.25 | DB=1.2754
min_cluster=50 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan


,representation,algorithm,min_cluster_size,min_samples,n_clusters,n_noise,noise_fraction,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n,elapsed_seconds
0,audio,HDBSCAN_PCA50,30,5,2,1317,0.6109,0.1060,30.5981,1.5382,31.0,808.0,0.4194
1,audio,HDBSCAN_PCA50,50,5,0,2156,1.0000,NaN,NaN,NaN,NaN,NaN,0.3508
2,audio,HDBSCAN_PCA50,100,5,0,2156,1.0000,NaN,NaN,NaN,NaN,NaN,0.3477
3,text,HDBSCAN_PCA50,30,5,5,1595,0.7398,0.2716,97.8483,1.2724,31.0,379.0,0.4665
4,text,HDBSCAN_PCA50,50,5,2,1265,0.5867,0.2514,111.4081,1.2839,65.0,826.0,0.4905
5,text,HDBSCAN_PCA50,100,5,0,2156,1.0000,NaN,NaN,NaN,NaN,NaN,0.4768
6,fusion,HDBSCAN_PCA50,30,5,3,1672,0.7755,0.2584,104.2464,1.2754,47.0,389.0,0.4485
7,fusion,HDBSCAN_PCA50,50,5,0,2156,1.0000,NaN,NaN,NaN,NaN,NaN,0.4236
8,fusion,HDBSCAN_PCA50,100,5,0,2156,1.0000,NaN,NaN,NaN,NaN,NaN,0.4856


18.15 — Setup stabilità tramite balanced resampling

In [16]:
from sklearn.metrics import adjusted_rand_score


def balanced_sample_indices(
    metadata,
    max_turns_per_patient=25,
    seed=42
):

    rng = np.random.default_rng(seed)

    selected = []

    for patient_id, group in metadata.groupby(
        "patient_id",
        sort=False
    ):

        indices = group.index.to_numpy()

        if len(indices) > max_turns_per_patient:

            indices = rng.choice(
                indices,
                size=max_turns_per_patient,
                replace=False
            )

        selected.extend(indices)

    return np.sort(
        np.asarray(selected)
    )


# PCA già apprese nella pipeline principale
PCA_MODELS = {
    "audio": pca_audio,
    "text": pca_text,
    "fusion": pca_fusion
}


FULL_REPRESENTATIONS = {
    "audio": X_audio,
    "text": X_text,
    "fusion": X_fusion
}


print(
    "✓ Setup stabilità completato."
)

✓ Setup stabilità completato.


18.16 — Label di riferimento

In [17]:
reference_turn_ids = (
    balanced_metadata[
        "turn_id"
    ]
    .astype(str)
    .to_numpy()
)


REFERENCE_LABELS = {}


for representation in [
    "audio",
    "text",
    "fusion"
]:

    REFERENCE_LABELS[
        (representation, "KMeans")
    ] = pd.Series(
        kmeans_labels[
            (representation, 2)
        ],
        index=reference_turn_ids
    )

    REFERENCE_LABELS[
        (representation, "Ward")
    ] = pd.Series(
        hierarchical_labels[
            (representation, 2)
        ],
        index=reference_turn_ids
    )


print(
    "Turni nel riferimento:",
    len(reference_turn_ids)
)

print(
    "✓ Label di riferimento preparate."
)

Turni nel riferimento: 2156
✓ Label di riferimento preparate.


18.17 — Stabilità ARI

In [18]:
N_RESAMPLES = 20

stability_rows = []


for seed in range(
    100,
    100 + N_RESAMPLES
):

    repeat_indices = balanced_sample_indices(
        metadata_df,
        max_turns_per_patient=25,
        seed=seed
    )

    repeat_metadata = (
        metadata_df
        .loc[repeat_indices]
        .copy()
    )

    repeat_turn_ids = (
        repeat_metadata[
            "turn_id"
        ]
        .astype(str)
        .to_numpy()
    )


    for representation in [
        "audio",
        "text",
        "fusion"
    ]:

        # Trasformazione nello stesso spazio PCA
        X_repeat = PCA_MODELS[
            representation
        ].transform(
            FULL_REPRESENTATIONS[
                representation
            ][repeat_indices]
        )


        # -----------------------------------------
        # KMeans K=2
        # -----------------------------------------

        km = KMeans(
            n_clusters=2,
            random_state=42,
            n_init=50
        )

        labels_km = km.fit_predict(
            X_repeat
        )


        # -----------------------------------------
        # Ward K=2
        # -----------------------------------------

        ward = AgglomerativeClustering(
            n_clusters=2,
            linkage="ward"
        )

        labels_ward = ward.fit_predict(
            X_repeat
        )


        current_labels = {
            "KMeans":
                pd.Series(
                    labels_km,
                    index=repeat_turn_ids
                ),

            "Ward":
                pd.Series(
                    labels_ward,
                    index=repeat_turn_ids
                )
        }


        for algorithm in [
            "KMeans",
            "Ward"
        ]:

            reference = REFERENCE_LABELS[
                (
                    representation,
                    algorithm
                )
            ]

            current = current_labels[
                algorithm
            ]


            common_turns = (
                reference.index
                .intersection(
                    current.index
                )
            )


            ari = adjusted_rand_score(
                reference.loc[
                    common_turns
                ],
                current.loc[
                    common_turns
                ]
            )


            if algorithm == "KMeans":

                silhouette = silhouette_score(
                    X_repeat,
                    labels_km
                )

            else:

                silhouette = silhouette_score(
                    X_repeat,
                    labels_ward
                )


            stability_rows.append(
                {
                    "seed":
                        seed,

                    "representation":
                        representation,

                    "algorithm":
                        algorithm,

                    "k":
                        2,

                    "n_turns":
                        len(repeat_indices),

                    "n_overlap":
                        len(common_turns),

                    "overlap_fraction":
                        len(common_turns)
                        / len(reference),

                    "ARI":
                        ari,

                    "silhouette":
                        silhouette
                }
            )

stability_df = pd.DataFrame(
    stability_rows
)


print(
    "Esperimenti completati:",
    len(stability_df)
)

display(
    stability_df.head()
)

Esperimenti completati: 120


,seed,representation,algorithm,k,n_turns,n_overlap,overlap_fraction,ARI,silhouette
0,100,audio,KMeans,2,2156,1418,0.657699,0.960446,0.241102
1,100,audio,Ward,2,2156,1418,0.657699,0.220974,0.210899
2,100,text,KMeans,2,2156,1418,0.657699,0.906313,0.080548
3,100,text,Ward,2,2156,1418,0.657699,-0.069900,0.091217
4,100,fusion,KMeans,2,2156,1418,0.657699,0.957998,0.111642


18.18 — Riepilogo stabilità

In [19]:
stability_summary = (
    stability_df
    .groupby(
        [
            "representation",
            "algorithm"
        ]
    )
    .agg(
        ARI_mean=(
            "ARI",
            "mean"
        ),
        ARI_std=(
            "ARI",
            "std"
        ),
        ARI_min=(
            "ARI",
            "min"
        ),
        ARI_max=(
            "ARI",
            "max"
        ),
        silhouette_mean=(
            "silhouette",
            "mean"
        ),
        silhouette_std=(
            "silhouette",
            "std"
        ),
        overlap_mean=(
            "overlap_fraction",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "ARI_mean",
        ascending=False
    )
)


display(
    stability_summary.round(4)
)

,representation,algorithm,ARI_mean,ARI_std,ARI_min,ARI_max,silhouette_mean,silhouette_std,overlap_mean
0,audio,KMeans,0.9899,0.0158,0.9377,1.0000,0.2409,0.0035,0.6518
2,fusion,KMeans,0.9493,0.0310,0.8995,0.9885,0.1114,0.0023,0.6518
4,text,KMeans,0.9254,0.0315,0.8591,0.9743,0.0787,0.0026,0.6518
1,audio,Ward,0.4418,0.2864,0.1143,0.9151,0.2271,0.0200,0.6518
3,fusion,Ward,0.2883,0.1657,0.0637,0.5903,0.1001,0.0206,0.6518
5,text,Ward,0.0592,0.1750,-0.0699,0.4598,0.0670,0.0276,0.6518


18.19 — Clustering finale Audio + KMeans K=2

In [20]:
FINAL_REPRESENTATION = "audio"
FINAL_ALGORITHM = "KMeans"
FINAL_K = 2


# PCA audio appresa sul campione bilanciato
X_audio_all_pca = pca_audio.transform(
    X_audio
)


# KMeans finale già addestrato sul campione bilanciato
final_kmeans = kmeans_models[
    ("audio", 2)
]


# Assegnazione di tutti i 3710 turni
final_labels_all = final_kmeans.predict(
    X_audio_all_pca
)


print(
    "Turni assegnati:",
    len(final_labels_all)
)

print(
    "Cluster:",
    np.unique(
        final_labels_all,
        return_counts=True
    )
)

Turni assegnati: 3710
Cluster: (array([0, 1], dtype=int32), array([1433, 2277]))


18.20 — Dataset turn-level con cluster audio

In [21]:
final_turn_clusters_df = (
    metadata_df.copy()
)


final_turn_clusters_df[
    "audio_cluster_k2"
] = final_labels_all


display(
    final_turn_clusters_df.head()
)


print(
    "\nDistribuzione globale:"
)

display(
    final_turn_clusters_df[
        "audio_cluster_k2"
    ]
    .value_counts()
    .sort_index()
)

,patient_id,recording_id,turn_id,turn_index,turn_duration_seconds,audio_path_turn,transcript_clean,audio_cluster_k2
0,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006__patient_turn_0000,0,1.771000,c:\Users\acer\Desktop\ProgettoTesi\risultati\f...,"e ti guarda, ti guarda, sei",1
1,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006__patient_turn_0001,1,1.975062,c:\Users\acer\Desktop\ProgettoTesi\risultati\f...,il 30 novembre.,1
2,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006__patient_turn_0002,2,15.963000,c:\Users\acer\Desktop\ProgettoTesi\risultati\f...,ho fatto delle chemie ho fatto sia la domanda ...,0
3,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006__patient_turn_0003,3,3.830000,c:\Users\acer\Desktop\ProgettoTesi\risultati\f...,Io ero specializzato in macchina in momento da...,0
4,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006__patient_turn_0005,5,1.857063,c:\Users\acer\Desktop\ProgettoTesi\risultati\f...,il quattro,1



Distribuzione globale:


audio_cluster_k2
0    1433
1    2277
Name: count, dtype: int64

18.21 — Proporzioni cluster per paziente

In [22]:
patient_cluster_counts = (
    final_turn_clusters_df
    .groupby(
        [
            "patient_id",
            "audio_cluster_k2"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


# Garantiamo entrambe le colonne
for cluster in [0, 1]:

    if cluster not in patient_cluster_counts.columns:

        patient_cluster_counts[
            cluster
        ] = 0


patient_cluster_counts = (
    patient_cluster_counts[
        [0, 1]
    ]
)


patient_cluster_counts.columns = [
    "n_cluster_0",
    "n_cluster_1"
]


patient_cluster_counts[
    "n_turns"
] = (
    patient_cluster_counts[
        "n_cluster_0"
    ]
    +
    patient_cluster_counts[
        "n_cluster_1"
    ]
)


patient_cluster_counts[
    "prop_cluster_0"
] = (
    patient_cluster_counts[
        "n_cluster_0"
    ]
    /
    patient_cluster_counts[
        "n_turns"
    ]
)


patient_cluster_counts[
    "prop_cluster_1"
] = (
    patient_cluster_counts[
        "n_cluster_1"
    ]
    /
    patient_cluster_counts[
        "n_turns"
    ]
)


patient_cluster_profiles_df = (
    patient_cluster_counts
    .reset_index()
)


display(
    patient_cluster_profiles_df.head()
)


print(
    "\nProporzione cluster 1:"
)

display(
    patient_cluster_profiles_df[
        "prop_cluster_1"
    ].describe()
)

,patient_id,n_cluster_0,n_cluster_1,n_turns,prop_cluster_0,prop_cluster_1
0,1,21,11,32,0.656250,0.343750
1,2,11,18,29,0.379310,0.620690
2,3,17,10,27,0.629630,0.370370
3,4,21,11,32,0.656250,0.343750
4,10,7,15,22,0.318182,0.681818



Proporzione cluster 1:


count    90.000000
mean      0.637131
std       0.172905
min       0.260000
25%       0.500000
50%       0.633399
75%       0.780707
max       0.968750
Name: prop_cluster_1, dtype: float64

18.22 — Profili misti per paziente

In [23]:
patient_cluster_profiles_df[
    "dominant_cluster"
] = np.where(
    patient_cluster_profiles_df[
        "prop_cluster_1"
    ] >= 0.5,
    1,
    0
)


patient_cluster_profiles_df[
    "mixed_clusters"
] = (
    (
        patient_cluster_profiles_df[
            "n_cluster_0"
        ] > 0
    )
    &
    (
        patient_cluster_profiles_df[
            "n_cluster_1"
        ] > 0
    )
)


print(
    "Pazienti totali:",
    len(
        patient_cluster_profiles_df
    )
)

print(
    "Pazienti con entrambi i cluster:",
    int(
        patient_cluster_profiles_df[
            "mixed_clusters"
        ].sum()
    )
)


print(
    "\nCluster dominante:"
)

display(
    patient_cluster_profiles_df[
        "dominant_cluster"
    ]
    .value_counts()
    .sort_index()
)

Pazienti totali: 90
Pazienti con entrambi i cluster: 90

Cluster dominante:


dominant_cluster
0    21
1    69
Name: count, dtype: int64

18.23 — Salvataggio

In [24]:
final_turn_clusters_df.to_csv(
    CLUSTER_DIR
    / "turni_cluster_audio_kmeans_k2.csv",
    index=False
)


patient_cluster_profiles_df.to_csv(
    CLUSTER_DIR
    / "profili_cluster_audio_per_paziente.csv",
    index=False
)


kmeans_results_df.to_csv(
    CLUSTER_DIR
    / "risultati_kmeans.csv",
    index=False
)


gmm_results_df.to_csv(
    CLUSTER_DIR
    / "risultati_gmm.csv",
    index=False
)


hierarchical_results_df.to_csv(
    CLUSTER_DIR
    / "risultati_gerarchico.csv",
    index=False
)


hdbscan_results_df.to_csv(
    CLUSTER_DIR
    / "risultati_hdbscan_pca95.csv",
    index=False
)


hdbscan_pca50_results_df.to_csv(
    CLUSTER_DIR
    / "risultati_hdbscan_sensitivity_pca50.csv",
    index=False
)


stability_df.to_csv(
    CLUSTER_DIR
    / "stabilita_resampling_ARI.csv",
    index=False
)


stability_summary.to_csv(
    CLUSTER_DIR
    / "riepilogo_stabilita_ARI.csv",
    index=False
)


print(
    "✓ Risultati Notebook 18 salvati in:"
)

print(
    CLUSTER_DIR
)

✓ Risultati Notebook 18 salvati in:
C:\Users\acer\Desktop\ProgettoTesi\risultati\clustering_embedding_multimodali


18.23 — Audit durata / lunghezza trascrizione per cluster

In [26]:
from scipy.stats import mannwhitneyu, pointbiserialr

audit_df = final_turn_clusters_df.copy()

audit_df["n_words"] = (
    audit_df["transcript_clean"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

audit_df["log_duration"] = np.log1p(
    audit_df["turn_duration_seconds"]
)


# ------------------------------------------------------------
# Descrittive
# ------------------------------------------------------------

cluster_duration_summary = (
    audit_df
    .groupby("audio_cluster_k2")
    .agg(
        n_turns=("turn_id", "count"),

        duration_mean=(
            "turn_duration_seconds",
            "mean"
        ),

        duration_median=(
            "turn_duration_seconds",
            "median"
        ),

        duration_std=(
            "turn_duration_seconds",
            "std"
        ),

        words_mean=(
            "n_words",
            "mean"
        ),

        words_median=(
            "n_words",
            "median"
        )
    )
    .reset_index()
)

display(
    cluster_duration_summary.round(3)
)


# ------------------------------------------------------------
# Mann-Whitney sulla durata
# ------------------------------------------------------------

duration_0 = audit_df.loc[
    audit_df["audio_cluster_k2"] == 0,
    "turn_duration_seconds"
]

duration_1 = audit_df.loc[
    audit_df["audio_cluster_k2"] == 1,
    "turn_duration_seconds"
]


u_duration, p_duration = mannwhitneyu(
    duration_0,
    duration_1,
    alternative="two-sided"
)


# Rank-biserial effect size
rank_biserial_duration = (
    2 * u_duration
    / (len(duration_0) * len(duration_1))
    - 1
)


# ------------------------------------------------------------
# Correlazione point-biserial cluster-durata
# ------------------------------------------------------------

r_duration, p_r_duration = pointbiserialr(
    audit_df["audio_cluster_k2"],
    audit_df["log_duration"]
)


r_words, p_r_words = pointbiserialr(
    audit_df["audio_cluster_k2"],
    audit_df["n_words"]
)


print("\n===== ASSOCIAZIONE CON DURATA =====")

print(
    "Mann-Whitney p:",
    f"{p_duration:.3e}"
)

print(
    "Rank-biserial:",
    round(rank_biserial_duration, 4)
)

print(
    "Point-biserial cluster ~ log(durata):",
    round(r_duration, 4),
    "| p =",
    f"{p_r_duration:.3e}"
)

print(
    "Point-biserial cluster ~ n_words:",
    round(r_words, 4),
    "| p =",
    f"{p_r_words:.3e}"
)

,audio_cluster_k2,n_turns,duration_mean,duration_median,duration_std,words_mean,words_median
0,0,1433,5.798,4.050,4.916,12.465,9.0
1,1,2277,1.222,1.114,0.506,3.036,3.0



===== ASSOCIAZIONE CON DURATA =====
Mann-Whitney p: 0.000e+00
Rank-biserial: 0.9938
Point-biserial cluster ~ log(durata): -0.8025 | p = 0.000e+00
Point-biserial cluster ~ n_words: -0.5317 | p = 7.574e-270


18.25 — Cluster per quartile di durata

In [27]:
audit_df["duration_quartile"] = pd.qcut(
    audit_df["turn_duration_seconds"],
    q=4,
    labels=[
        "Q1_shortest",
        "Q2",
        "Q3",
        "Q4_longest"
    ],
    duplicates="drop"
)


duration_cluster_table = pd.crosstab(
    audit_df["duration_quartile"],
    audit_df["audio_cluster_k2"],
    normalize="index"
)


display(
    duration_cluster_table.round(3)
)

audio_cluster_k2,0,1
duration_quartile,,
Q1_shortest,0.000,1.000
Q2,0.001,0.999
Q3,0.544,0.456
Q4_longest,1.000,0.000


18.26 — Rimozione dell'effetto della durata dagli embedding

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA


# ------------------------------------------------------------
# Variabile di confondimento
# ------------------------------------------------------------

log_duration_all = np.log1p(
    metadata_df["turn_duration_seconds"].to_numpy()
)

Z_all = np.column_stack(
    [
        log_duration_all,
        log_duration_all ** 2
    ]
)

Z_bal = Z_all[
    balanced_indices
]


# ------------------------------------------------------------
# Funzione: residualizzazione + nuova PCA
# ------------------------------------------------------------

def residualize_duration_and_pca(
    X_all,
    name
):

    X_bal = X_all[
        balanced_indices
    ]

    nuisance_model = LinearRegression()

    nuisance_model.fit(
        Z_bal,
        X_bal
    )

    # Rimuoviamo la componente prevedibile dalla durata
    X_residual_all = (
        X_all
        -
        nuisance_model.predict(
            Z_all
        )
    )

    X_residual_bal = (
        X_residual_all[
            balanced_indices
        ]
    )


    # Nuova PCA DOPO il controllo della durata
    pca = PCA(
        n_components=0.95,
        svd_solver="full"
    )

    X_residual_pca_bal = (
        pca.fit_transform(
            X_residual_bal
        )
    )


    print(
        f"{name:10s} | "
        f"{X_all.shape[1]} -> "
        f"{X_residual_pca_bal.shape[1]} componenti | "
        f"varianza={pca.explained_variance_ratio_.sum():.4f}"
    )


    return {
        "regressor": nuisance_model,
        "residual_all": X_residual_all,
        "residual_bal": X_residual_bal,
        "pca": pca,
        "pca_bal": X_residual_pca_bal
    }


duration_controlled = {
    "audio":
        residualize_duration_and_pca(
            X_audio,
            "AUDIO"
        ),

    "text":
        residualize_duration_and_pca(
            X_text,
            "TESTO"
        ),

    "fusion":
        residualize_duration_and_pca(
            X_fusion,
            "FUSION"
        )
}

AUDIO      | 768 -> 350 componenti | varianza=0.9501
TESTO      | 384 -> 140 componenti | varianza=0.9506
FUSION     | 1152 -> 288 componenti | varianza=0.9501


18.27 — KMeans dopo residualizzazione della durata

In [29]:
CONTROLLED_REPRESENTATIONS = {
    name: data["pca_bal"]
    for name, data
    in duration_controlled.items()
}


controlled_kmeans_results = []
controlled_kmeans_labels = {}
controlled_kmeans_models = {}


for representation, X in CONTROLLED_REPRESENTATIONS.items():

    print(
        f"\n===== KMEANS DURATION-CONTROLLED | "
        f"{representation.upper()} ====="
    )

    for k in range(2, 7):

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=50
        )

        labels = model.fit_predict(
            X
        )

        metrics = evaluate_clustering(
            X=X,
            labels=labels,
            representation=representation,
            algorithm="KMeans_duration_controlled",
            k=k
        )

        controlled_kmeans_results.append(
            metrics
        )

        controlled_kmeans_labels[
            (representation, k)
        ] = labels

        controlled_kmeans_models[
            (representation, k)
        ] = model


        print(
            f"K={k} | "
            f"sil={metrics['silhouette']:.4f} | "
            f"CH={metrics['calinski_harabasz']:.2f} | "
            f"DB={metrics['davies_bouldin']:.4f} | "
            f"min={metrics['smallest_cluster_n']} | "
            f"max={metrics['largest_cluster_n']}"
        )


controlled_kmeans_results_df = pd.DataFrame(
    controlled_kmeans_results
)


===== KMEANS DURATION-CONTROLLED | AUDIO =====
K=2 | sil=0.0512 | CH=128.42 | DB=3.9957 | min=979 | max=1177
K=3 | sil=0.0531 | CH=118.56 | DB=3.5463 | min=512 | max=879
K=4 | sil=0.0498 | CH=106.01 | DB=3.3988 | min=253 | max=660
K=5 | sil=0.0537 | CH=94.83 | DB=3.2298 | min=117 | max=594
K=6 | sil=0.0525 | CH=83.51 | DB=3.3023 | min=89 | max=545

===== KMEANS DURATION-CONTROLLED | TEXT =====
K=2 | sil=0.0655 | CH=98.34 | DB=4.2858 | min=639 | max=1517
K=3 | sil=0.0531 | CH=94.84 | DB=3.9655 | min=454 | max=1014
K=4 | sil=0.0606 | CH=85.72 | DB=3.5861 | min=215 | max=1003
K=5 | sil=0.0522 | CH=81.36 | DB=3.5201 | min=180 | max=759
K=6 | sil=0.0466 | CH=75.58 | DB=3.4737 | min=155 | max=579

===== KMEANS DURATION-CONTROLLED | FUSION =====
K=2 | sil=0.0444 | CH=83.40 | DB=4.9897 | min=889 | max=1267
K=3 | sil=0.0445 | CH=79.83 | DB=4.3176 | min=452 | max=1031
K=4 | sil=0.0419 | CH=72.18 | DB=4.1323 | min=299 | max=846
K=5 | sil=0.0439 | CH=67.88 | DB=3.9222 | min=180 | max=768
K=6 | si

18.28 — Ranking dopo controllo durata

In [30]:
controlled_ranking = (
    controlled_kmeans_results_df
    [
        [
            "representation",
            "algorithm",
            "k",
            "silhouette",
            "calinski_harabasz",
            "davies_bouldin",
            "smallest_cluster_n",
            "largest_cluster_n"
        ]
    ]
    .sort_values(
        "silhouette",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    controlled_ranking.round(4)
)

,representation,algorithm,k,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n
0,text,KMeans_duration_controlled,2,0.0655,98.3387,4.2858,639,1517
1,text,KMeans_duration_controlled,4,0.0606,85.7206,3.5861,215,1003
2,audio,KMeans_duration_controlled,5,0.0537,94.8255,3.2298,117,594
3,audio,KMeans_duration_controlled,3,0.0531,118.5605,3.5463,512,879
4,text,KMeans_duration_controlled,3,0.0531,94.8395,3.9655,454,1014
5,audio,KMeans_duration_controlled,6,0.0525,83.5069,3.3023,89,545
6,text,KMeans_duration_controlled,5,0.0522,81.3563,3.5201,180,759
7,audio,KMeans_duration_controlled,2,0.0512,128.4179,3.9957,979,1177
8,audio,KMeans_duration_controlled,4,0.0498,106.0148,3.3988,253,660
9,text,KMeans_duration_controlled,6,0.0466,75.5826,3.4737,155,579


18.29 — Stabilità KMeans K=2 dopo controllo durata

In [31]:
controlled_reference_labels = {}

for representation in [
    "audio",
    "text",
    "fusion"
]:

    controlled_reference_labels[
        representation
    ] = pd.Series(
        controlled_kmeans_labels[
            (representation, 2)
        ],
        index=reference_turn_ids
    )


controlled_stability_rows = []


for seed in range(100, 120):

    repeat_indices = balanced_sample_indices(
        metadata_df,
        max_turns_per_patient=25,
        seed=seed
    )

    repeat_turn_ids = (
        metadata_df
        .loc[repeat_indices, "turn_id"]
        .astype(str)
        .to_numpy()
    )


    for representation in [
        "audio",
        "text",
        "fusion"
    ]:

        data = duration_controlled[
            representation
        ]

        # Residui già calcolati su tutti i 3710 turni
        X_repeat_residual = (
            data["residual_all"][
                repeat_indices
            ]
        )

        # Stessa PCA della rappresentazione
        # duration-controlled di riferimento
        X_repeat_pca = (
            data["pca"].transform(
                X_repeat_residual
            )
        )


        model = KMeans(
            n_clusters=2,
            random_state=42,
            n_init=50
        )

        labels = model.fit_predict(
            X_repeat_pca
        )


        current = pd.Series(
            labels,
            index=repeat_turn_ids
        )

        reference = (
            controlled_reference_labels[
                representation
            ]
        )


        common_turns = (
            reference.index
            .intersection(
                current.index
            )
        )


        ari = adjusted_rand_score(
            reference.loc[
                common_turns
            ],
            current.loc[
                common_turns
            ]
        )


        sil = silhouette_score(
            X_repeat_pca,
            labels
        )


        controlled_stability_rows.append(
            {
                "seed":
                    seed,

                "representation":
                    representation,

                "algorithm":
                    "KMeans_duration_controlled",

                "k":
                    2,

                "n_overlap":
                    len(common_turns),

                "ARI":
                    ari,

                "silhouette":
                    sil
            }
        )


controlled_stability_df = pd.DataFrame(
    controlled_stability_rows
)


print(
    "Esperimenti:",
    len(controlled_stability_df)
)

Esperimenti: 60


18.30 — Riepilogo stabilità duration-controlled

In [32]:
controlled_stability_summary = (
    controlled_stability_df
    .groupby(
        "representation"
    )
    .agg(
        ARI_mean=("ARI", "mean"),
        ARI_std=("ARI", "std"),
        ARI_min=("ARI", "min"),
        ARI_max=("ARI", "max"),
        silhouette_mean=("silhouette", "mean"),
        silhouette_std=("silhouette", "std")
    )
    .reset_index()
    .sort_values(
        "ARI_mean",
        ascending=False
    )
)


display(
    controlled_stability_summary.round(4)
)

,representation,ARI_mean,ARI_std,ARI_min,ARI_max,silhouette_mean,silhouette_std
2,text,0.8478,0.0670,0.7493,0.9571,0.0655,0.0026
0,audio,0.6422,0.2019,0.3486,0.9603,0.0555,0.0030
1,fusion,0.0395,0.1934,-0.0067,0.8611,0.0500,0.0028


In [33]:
controlled_kmeans_results_df.to_csv(
    CLUSTER_DIR
    / "risultati_kmeans_duration_controlled.csv",
    index=False
)

controlled_stability_df.to_csv(
    CLUSTER_DIR
    / "stabilita_duration_controlled.csv",
    index=False
)

controlled_stability_summary.to_csv(
    CLUSTER_DIR
    / "riepilogo_stabilita_duration_controlled.csv",
    index=False
)

print("✓ Sensitivity duration-controlled salvata.")

✓ Sensitivity duration-controlled salvata.
